# Exercise 7: KNNRegressor and RMSE on cpu.csv

In [1]:
import sys
from pathlib import Path

import numpy as np

# find the repository root (the folder that contains "datasets"), wherever the notebook is
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'datasets').is_dir())
for path in (ROOT / 'src', ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

DATA = ROOT / 'datasets'

from si.io.csv_file import read_csv
from si.model_selection.split import train_test_split
from si.models.knn_regressor import KNNRegressor
from si.metrics.rmse import rmse

In [2]:
cpu = read_csv(str(DATA / 'cpu' / 'cpu.csv'), features=True, label=True)
print(cpu.shape(), cpu.features, '->', cpu.label)
train, test = train_test_split(cpu, test_size=0.2, random_state=42)

(209, 6) ['syct', 'mmin', 'mmax', 'cach', 'chmin', 'chmax'] -> perf


In [3]:
knn = KNNRegressor(k=3)
knn.fit(train)
print('RMSE (test):', round(knn.score(test), 4))
print('first predictions:', knn.predict(test)[:5].round(1))
print('real values:      ', test.y[:5])

RMSE (test): 59.3059
first predictions: [ 17.7 447.3  68.7  30.7 307. ]
real values:       [ 24 489  24  38 370]


## Effect of k
Small k follows the training data closely (high variance); large k averages many neighbours (smoother, higher bias).

In [4]:
for k in [1, 3, 5, 10, 20]:
    print(k, round(KNNRegressor(k=k).fit(train).score(test), 2))

1 98.92
3 59.31
5 85.72
10 112.68
20 142.45


## Baseline
Predicting the mean of y for every sample. The model is only useful if its RMSE is lower than this.

In [5]:
baseline = np.full(test.shape()[0], train.y.mean())
print('baseline RMSE:', round(rmse(test.y, baseline), 2))

baseline RMSE: 213.92
